In [21]:
import Pkg
Pkg.activate(".")
Pkg.add([
    "Oceananigans",
    "NumericalEarth",
    "CUDA",
    "CairoMakie",
    "JLD2"
])

Pkg.precompile()

  Activating project at `c:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\simulation #1`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\simulation #1\Project.toml`
    Manifest No packages added to or removed from `C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\simulation #1\Manifest.toml`


In [22]:

using NumericalEarth
using Oceananigans
using Oceananigans.Units
using Dates
using Printf
using Statistics
using CUDA

using Oceananigans.Models: add_tracer!

In [ ]:

arch = GPU()

# grid definitions -- amazon river mouth / plume region


long_west = -58.5
long_east = -41.5
lat_south = -7.8
lat_north = 9.2

Nx = 170
Ny = 170
Nz = 20

long_river = -49.5
lat_river  = 0.16

depth = 4000meters # decreased depth to 4k for this test 

z = ExponentialDiscretization(Nz, -depth, 0; scale = depth/4, mutable = false)
underlying_grid = LatitudeLongitudeGrid(
    arch;
    size = (Nx, Ny, Nz),
    halo = (5, 5, 4),
    longitude = (long_west, long_east),
    latitude = (lat_south, lat_north),
    z,
    topology = (Bounded, Bounded, Bounded)
)

bottom_height = regrid_bathymetry(underlying_grid;
                                  minimum_depth = 10,
                                  interpolation_passes = 7, # reduce num passes since we want to maintain some more specific bathymetry features in the reg. grid? 
                                  major_basins = 1) # only one major basin here -- atlantic 

grid = ImmersedBoundaryGrid(underlying_grid, GridFittedBottom(bottom_height);
                            active_cells_map=true)

[ Info: Interpolation passes of bathymetry size (21600, 10800, 1) onto a LatitudeLongitudeGrid target grid of size (180, 180, 20):
[ Info:     pass 1 to size (18540, 9283, 1)
[ Info:     pass 2 to size (15480, 7766, 1)
[ Info:     pass 3 to size (12420, 6249, 1)
[ Info:     pass 4 to size (9360, 4732, 1)
[ Info:     pass 5 to size (6300, 3215, 1)
[ Info:     pass 6 to size (3240, 1698, 1)
[ Info:     pass 7 to size (180, 180, 1)
[ Info: Saved bathymetry cache to C:\Users\meghn\.julia\scratchspaces\904d977b-046a-4731-8b86-9235c0d1ef02\bathymetry_cache\bathymetry_180x180_-58.5_-41.5_-7.8_9.2_3038b715.jld2


180×180×20 ImmersedBoundaryGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 5×5×4 halo:
├── immersed_boundary: GridFittedBottom(mean(z)=-1065.2, min(z)=-4000.0, max(z)=0.0)
├── underlying_grid: 180×180×20 LatitudeLongitudeGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 5×5×4 halo
├── longitude: Bounded  λ ∈ [-58.5, -41.5] regularly spaced with Δλ=0.0944444
├── latitude:  Bounded  φ ∈ [-7.8, 9.2]    regularly spaced with Δφ=0.0944444
└── z:         Bounded  z ∈ [-4000.0, 0.0] variably spaced with min(Δz)=16.5232, max(Δz)=738.605

In [24]:

using Oceananigans.TurbulenceClosures: IsopycnalSkewSymmetricDiffusivity, AdvectiveFormulation

eddy_closure = IsopycnalSkewSymmetricDiffusivity(
    κ_skew = 1e3,
    κ_symmetric = 1e3,
    skew_flux_formulation = AdvectiveFormulation()
)

vertical_mixing = NumericalEarth.Oceans.default_ocean_closure()

CATKEVerticalDiffusivity{VerticallyImplicitTimeDiscretization}
├── maximum_tracer_diffusivity: Inf
├── maximum_tke_diffusivity: Inf
├── maximum_viscosity: Inf
├── minimum_tke: 1.0e-9
├── negative_tke_time_scale: 60.0
├── minimum_convective_buoyancy_flux: 1.0e-11
├── tke_time_step: Nothing
├── mixing_length: TKEBasedVerticalDiffusivities.CATKEMixingLength
│   ├── Cˢ:   1.131
│   ├── Cᵇ:   0.01
│   ├── Cʰⁱu: 0.242
│   ├── Cʰⁱc: 0.098
│   ├── Cʰⁱe: 0.548
│   ├── Cˡᵒu: 0.361
│   ├── Cˡᵒc: 0.369
│   ├── Cˡᵒe: 7.863
│   ├── Cᵘⁿu: 0.37
│   ├── Cᵘⁿc: 0.572
│   ├── Cᵘⁿe: 1.447
│   ├── Cᶜu:  3.705
│   ├── Cᶜc:  4.793
│   ├── Cᶜe:  3.642
│   ├── Cᵉc:  0.112
│   ├── Cᵉe:  0.0
│   ├── Cˢᵖ:  0.505
│   ├── CRiᵟ: 1.02
│   └── CRi⁰: 0.254
└── turbulent_kinetic_energy_equation: TKEBasedVerticalDiffusivities.CATKEEquation
    ├── CʰⁱD: 0.579
    ├── CˡᵒD: 1.604
    ├── CᵘⁿD: 0.923
    ├── CᶜD:  3.254
    ├── CᵉD:  0.0
    ├── Cᵂu★: 3.179
    ├── CᵂwΔ: 0.383
    └── Cᵂϵ:  1.0

In [25]:
free_surface = SplitExplicitFreeSurface(grid; substeps = 70)
momentum_advection = WENOVectorInvariant(order = 5)
tracer_advection   = WENO(order = 5)


# create zero-gradient (Neumann) boundary conditions for all tracers
# "flow out" part of the model 
tracer_bcs = FieldBoundaryConditions(
    west   = GradientBoundaryCondition(0),
    east   = GradientBoundaryCondition(0),
    south  = GradientBoundaryCondition(0),
    north  = GradientBoundaryCondition(0),
    top    = FluxBoundaryCondition(0),
    bottom = FluxBoundaryCondition(0)
)

# compile tracer boundary conditions for the model
model_bcs = (
    # cant use t/s as neumann boundary tracers b/c atmospheric forcing --> unstable 
    # T   = tracer_bcs, 
    # S   = tracer_bcs,
    dye = tracer_bcs,
)

# added dye + boundary conditions to the model 
ocean = ocean_simulation(grid;
                         momentum_advection, tracer_advection, free_surface,
                         closure = (eddy_closure, vertical_mixing),
                         tracers = (:T, :S, :dye),
                         boundary_conditions = model_bcs)

# instead of floating dye patch, new dye patch centered on the river mouth, fading with distance
# dye is initially deposited only in the upper 10 meters
@inline function dye_initial_condition(x, y, z)
    horizontal_width = 0.5

    horizontal_blob = exp(-((x - long_river)^2 + (y - lat_river)^2) / horizontal_width^2)

    if z > -10
        return horizontal_blob
    else
        return 0
    end
end

# print the model structure so we can check that dye is listed as a tracer and that the boundary conditions were set to 0 gradient
@info "We've built an ocean simulation with model:"
@show ocean.model
@show ocean.model.tracers.dye.boundary_conditions

ocean.model = HydrostaticFreeSurfaceModel{CUDAGPU, ImmersedBoundaryGrid}(time = 0 seconds, iteration = 0)
├── grid: 180×180×20 ImmersedBoundaryGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 5×5×4 halo
├── timestepper: SplitRungeKuttaTimeStepper
├── tracers: (T, S, dye, e)
├── closure: Tuple with 2 closures:
│   ├── CATKEVerticalDiffusivity{VerticallyImplicitTimeDiscretization}
│   └── IsopycnalSkewSymmetricDiffusivity(κ_skew=1000.0, κ_symmetric=1000.0)
├── buoyancy: SeawaterBuoyancy with g=9.80665 and BoussinesqEquationOfState{Float64} with ĝ = NegativeZDirection()
├── free surface: SplitExplicitFreeSurface with gravitational acceleration 9.80665 m s⁻²
│   └── substepping: FixedSubstepNumber(50)
├── advection scheme: 
│   ├── momentum: WENOVectorInvariant{3, Float64}(vorticity_order=5, vertical_order=5)
│   ├── T: WENO{3, Float64, Oceananigans.Utils.BackendOptimizedDivision}(order=5)
│   ├── S: WENO{3, Float64, Oceananigans.Utils.BackendOptimizedDivision}(order=5)
│   ├── dy

[ Info: We've built an ocean simulation with model:


ocean.model.tracers.dye.boundary_conditions = Oceananigans.FieldBoundaryConditions, with boundary conditions
├── west: GradientBoundaryCondition: 0.0
├── east: GradientBoundaryCondition: 0.0
├── south: GradientBoundaryCondition: 0.0
├── north: GradientBoundaryCondition: 0.0
├── bottom: FluxBoundaryCondition: 0.0
├── top: FluxBoundaryCondition: 0.0
└── immersed: FluxBoundaryCondition: Nothing


Oceananigans.FieldBoundaryConditions, with boundary conditions
├── west: GradientBoundaryCondition: 0.0
├── east: GradientBoundaryCondition: 0.0
├── south: GradientBoundaryCondition: 0.0
├── north: GradientBoundaryCondition: 0.0
├── bottom: FluxBoundaryCondition: 0.0
├── top: FluxBoundaryCondition: 0.0
└── immersed: FluxBoundaryCondition: Nothing

In [ ]:
# ask for ecco credentials at runtime so they are not saved in the notebook


date = DateTime(1993, 1, 1)

ecco_variables = (:temperature, :salinity)
ecco_set = MetadataSet(ecco_variables; dataset = ECCO4Monthly(), date)

set!(ocean.model,   ecco_set)  

# initialize the dye tracer with initial condition function 
set!(ocean.model, dye = dye_initial_condition)

# print out all of the tracers 
@show keys(ocean.model.tracers)

keys(ocean.model.tracers) = (:T, :S, :dye, :e)


(:T, :S, :dye, :e)

In [27]:

land = JRA55PrescribedLand(arch)
atmosphere = JRA55PrescribedAtmosphere(arch)
ocean_surface = SurfaceRadiationProperties(albedo = LatitudeDependentAlbedo())
radiation = JRA55PrescribedRadiation(arch; ocean_surface)

640×320×1×2920 PrescribedRadiation on LatitudeLongitudeGrid:
├── times: 2920-element StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}
├── stefan_boltzmann_constant: 5.67037e-8
└── surface_properties: (:ocean, :sea_ice)

In [28]:
coupled_model = OceanSeaIceModel(ocean, nothing; atmosphere, land, radiation)

#model runs for 30 days with a timestep of 20 
simulation = Simulation(coupled_model; Δt=5minutes, stop_time=10days)

┌ Warning: 1440×720×1×365 PrescribedLand tracks time as Float32 but the EarthSystemModel clock uses Float64; coercing the component clock to keep components synchronized.
└ @ NumericalEarth.EarthSystemModels C:\Users\meghn\.julia\packages\NumericalEarth\eKhWQ\src\EarthSystemModels\components.jl:136


Simulation of EarthSystemModel{GPU}(time = 0 seconds, iteration = 0)
├── Next time step: 5 minutes
├── run_wall_time: 0 seconds
├── run_wall_time / iteration: NaN days
├── stop_time: 10 days
├── stop_iteration: Inf
├── wall_time_limit: Inf
├── minimum_relative_step: 0.0
├── callbacks: OrderedDict with 4 entries:
│   ├── stop_time_exceeded => Callback of stop_time_exceeded on IterationInterval(1)
│   ├── stop_iteration_exceeded => Callback of stop_iteration_exceeded on IterationInterval(1)
│   ├── wall_time_limit_exceeded => Callback of wall_time_limit_exceeded on IterationInterval(1)
│   └── nan_checker => Callback of NaNChecker for T_ocean on IterationInterval(100)
└── output_writers: OrderedDict with no entries

In [29]:

wall_time = Ref(time_ns())

function progress(sim)
 
    ocean = sim.model.ocean

    u, v, w = ocean.model.velocities

    T = ocean.model.tracers.T
    e = ocean.model.tracers.e

    # grab the dye tracer so it gets saved with the ocean output
    dye = ocean.model.tracers.dye

    Tmin, Tmax, Tavg = minimum(T), maximum(T), mean(view(T, :, :, ocean.model.grid.Nz))
    emax = maximum(e)

    # compute max speed components, so we can see if the model is behaving weirdly
    umax = (maximum(abs, u), maximum(abs, v), maximum(abs, w))

    step_time = 1e-9 * (time_ns() - wall_time[])

    msg1 = @sprintf("time: %s, iter: %d", prettytime(sim), iteration(sim))
    msg2 = @sprintf(", max|uo|: (%.1e, %.1e, %.1e) m s⁻¹", umax...)
    msg3 = @sprintf(", extrema(To): (%.1f, %.1f) ᵒC, mean(To(z=0)): %.1f ᵒC", Tmin, Tmax, Tavg)
    msg4 = @sprintf(", max(e): %.2f m² s⁻²", emax)
    msg5 = @sprintf(", wall time: %s \n", prettytime(step_time))

    @info msg1 * msg2 * msg3 * msg4 * msg5

    wall_time[] = time_ns()

     return nothing
end

progress (generic function with 1 method)

In [30]:

add_callback!(simulation, progress, TimeInterval(5days))

In [31]:
# this collects the ocean tracers + velocities into one output group
# tracers are T, S, e, and dye,  while velocities are u, v, and w

ocean_outputs = merge(ocean.model.tracers, ocean.model.velocities)

free_surface = ocean.model.free_surface.displacement

ocean.output_writers[:surface] = JLD2Writer(ocean.model, ocean_outputs;
                                            schedule = TimeInterval(1days),
                                            filename = "ocean_one_degree_surface_fields",
                                            indices = (:, :, grid.Nz),
                                            overwrite_existing = true)


ocean.output_writers[:free_surface] = JLD2Writer(ocean.model, (; η = free_surface);
                                                 schedule = TimeInterval(1days),
                                                 filename = "ocean_one_degree_free_surface",
                                                 overwrite_existing = true)



JLD2Writer scheduled on TimeInterval(1 day):
├── filepath: ocean_one_degree_free_surface.jld2
├── 1 outputs: η
├── array_type: Array{Float32}
├── including: [:coriolis, :buoyancy, :closure]
├── file_splitting: NoFileSplitting
└── file size: 0 bytes (file not yet created)

In [32]:

run!(simulation)

[ Info: Initializing simulation...
[ Info: time: 0 seconds, iter: 0, max|uo|: (0.0e+00, 0.0e+00, 0.0e+00) m s⁻¹, extrema(To): (2.1, 28.2) ᵒC, mean(To(z=0)): 27.0 ᵒC, max(e): 0.00 m² s⁻², wall time: 18.976 seconds 
[ Info:     ... simulation initialization complete (10.844 seconds)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (1.500 minutes).
[ Info: time: 5 days, iter: 1440, max|uo|: (7.6e-01, 5.4e-01, 1.6e-03) m s⁻¹, extrema(To): (2.1, 29.0) ᵒC, mean(To(z=0)): 27.1 ᵒC, max(e): 0.00 m² s⁻², wall time: 7.604 minutes 
[ Info: Simulation is stopping after running for 13.929 minutes.
[ Info: Simulation time 10 days equals or exceeds stop time 10 days.
[ Info: time: 10 days, iter: 2880, max|uo|: (8.7e-01, 5.0e-01, 2.4e-03) m s⁻¹, extrema(To): (2.1, 29.5) ᵒC, mean(To(z=0)): 27.1 ᵒC, max(e): 0.00 m² s⁻², wall time: 6.155 minutes 


In [33]:

using CairoMakie

uo = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "u"; backend = OnDisk())
vo = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "v"; backend = OnDisk())
To = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "T"; backend = OnDisk())
eo = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "e"; backend = OnDisk())
ηo = FieldTimeSeries("ocean_one_degree_free_surface.jld2",    "η"; backend = OnDisk())

# load dye tracer
dye = FieldTimeSeries("ocean_one_degree_surface_fields.jld2", "dye"; backend = OnDisk())

180×180×1×11 FieldTimeSeries{OnDisk} located at (Center, Center, Center) of dye at ocean_one_degree_surface_fields.jld2
├── grid: 180×180×20 ImmersedBoundaryGrid{Float64, Bounded, Bounded, Bounded} on CPU with 5×5×4 halo
├── indices: (:, :, 20:20)
├── time_indexing: Linear()
├── backend: OnDisk
├── path: ocean_one_degree_surface_fields.jld2
└── name: dye

In [34]:

times = uo.times
Nt = length(times)
n = Observable(Nt)

Observable(11)


In [53]:
# land mask from bathymetry
land_full = interior(To.grid.immersed_boundary.bottom_height) .≥ 0

Toₙ = @lift begin
    Tₙ = interior(To[$n])

    land = falses(size(Tₙ))
    i = min(size(Tₙ, 1), size(land_full, 1))
    j = min(size(Tₙ, 2), size(land_full, 2))
    k = min(size(Tₙ, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    Tₙ[land] .= NaN
    view(Tₙ, :, :, 1)
end

eoₙ = @lift begin
    eₙ = interior(eo[$n])

    land = falses(size(eₙ))
    i = min(size(eₙ, 1), size(land_full, 1))
    j = min(size(eₙ, 2), size(land_full, 2))
    k = min(size(eₙ, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    eₙ[land] .= NaN
    view(eₙ, :, :, 1)
end

ηoₙ = @lift begin
    ηₙ = interior(ηo[$n])

    land = falses(size(ηₙ))
    i = min(size(ηₙ, 1), size(land_full, 1))
    j = min(size(ηₙ, 2), size(land_full, 2))
    k = min(size(ηₙ, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    ηₙ[land] .= NaN
    view(ηₙ, :, :, 1)
end

dyeₙ = @lift begin
    d = interior(dye[$n])

    land = falses(size(d))
    i = min(size(d, 1), size(land_full, 1))
    j = min(size(d, 2), size(land_full, 2))
    k = min(size(d, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    d[land] .= NaN

    # log the values to make dye tracer more visible at smaller concentrations
    d_log = log10.(max.(d, 0) .+ 1e-16)

    view(d_log, :, :, 1)
end

uoₙ = Field{Face, Center, Nothing}(uo.grid)
voₙ = Field{Center, Face, Nothing}(vo.grid)
so = Field(sqrt(uoₙ^2 + voₙ^2))

soₙ = @lift begin
    parent(uoₙ) .= parent(uo[$n])
    parent(voₙ) .= parent(vo[$n])
    compute!(so)

    sₙ = interior(so)

    land = falses(size(sₙ))
    i = min(size(sₙ, 1), size(land_full, 1))
    j = min(size(sₙ, 2), size(land_full, 2))
    k = min(size(sₙ, 3), size(land_full, 3))
    land[1:i, 1:j, 1:k] .= land_full[1:i, 1:j, 1:k]

    sₙ[land] .= NaN
    view(sₙ, :, :, 1)
end

Observable([NaN NaN … 0.32591296222712207 0.1755022694280766; NaN NaN … 0.3685464333995368 0.4722817657111441; … ; NaN NaN … 0.10485099245508762 0.1080161018377083; 0.0 0.0 … 0.050921210063806885 0.03480047369133903])


In [54]:

fig = Figure(size=(1200, 1300))

title = @lift string("amazon river snapshot", prettytime(times[$n] - times[1]))

axso = Axis(fig[1, 1])
axηo = Axis(fig[1, 3])
axTo = Axis(fig[2, 1])
axeo = Axis(fig[2, 3])

# this adds one more panel for the passive dye tracer
axdye = Axis(fig[4, 1])


hm = heatmap!(axso, soₙ, colorrange = (0, 0.5), colormap = :deep,  nan_color=:lightgray)
Colorbar(fig[1, 2], hm, label = "Ocean Surface speed (m s⁻¹)")


hm = heatmap!(axηo, ηoₙ, colorrange = (-1.2, 1.2), colormap = :balance, nan_color=:lightgray)
Colorbar(fig[1, 4], hm, label = "Sea Surface Height (m)")


hm = heatmap!(axTo, Toₙ, colorrange = (-1, 32), colormap = :magma, nan_color=:lightgray)
Colorbar(fig[2, 2], hm, label = "Surface Temperature (ᵒC)")


hm = heatmap!(axeo, eoₙ, colorrange = (0, 1e-3), colormap = :solar, nan_color=:lightgray)
Colorbar(fig[2, 4], hm, label = "Turbulent Kinetic Energy (m² s⁻²)")


# passive dye tracer
hm = heatmap!(axdye, dyeₙ, colorrange = (-8, 0), colormap = :viridis, nan_color = :lightgray)
Colorbar(fig[4, 2], hm, label = "log₁₀(passive dye + 1e-16)")

for ax in (axso, axTo, axeo, axdye)
    hidedecorations!(ax)
end

Label(fig[0, :], title)


save("amazon_snapshot.png", fig)

In [55]:

CairoMakie.record(fig, "amazon_river.mp4", 1:Nt, framerate = 8) do nn
    n[] = nn
end

"amazon_river.mp4"